In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/featured_projects.csv")
print(df.shape)
df.head()

(3000, 22)


,project_id,project_name,project_type,state,district,land_area_hectares,affected_families,compensation_disbursed_pct,active_legal_disputes,possession_pct,...,dept_historical_delay_rate,days_since_notification,planned_completion_days,is_delayed,delay_days,compensation_pending_ratio,legal_dispute_density,rnr_gap,stage_velocity,family_density
0,LA-OD-00000,Irrigation Canal Project 0,Irrigation Canal,Odisha,Cuttack,55.4,295,52.7,1,49.1,...,0.67,338,432,0,0,47.3,0.338983,18.0,0.145266,5.324910
1,LA-GU-00001,Irrigation Canal Project 1,Irrigation Canal,Gujarat,Vadodara,73.5,156,28.3,0,68.2,...,0.67,240,230,1,1245,71.7,0.000000,34.3,0.284167,2.122449
2,LA-GU-00002,Irrigation Canal Project 2,Irrigation Canal,Gujarat,Vadodara,109.4,113,24.4,1,43.7,...,0.63,134,285,1,651,75.6,0.884956,38.8,0.326119,1.032907
3,LA-KA-00003,Irrigation Canal Project 3,Irrigation Canal,Karnataka,Belagavi,60.5,113,60.3,1,23.2,...,0.42,141,406,0,0,39.7,0.884956,20.2,0.164539,1.867769
4,LA-KA-00004,Rail Corridor Project 4,Rail Corridor,Karnataka,Bengaluru Rural,47.3,84,31.3,4,35.5,...,0.68,95,488,1,1203,68.7,4.761905,18.3,0.373684,1.775899


In [2]:
# Drop identifiers / non-feature columns.
# - project_id, project_name: sirf naam/ID hain, koi predictive signal nahi
# - district: high-cardinality hai (bahut zyada unique values), simple 
#   one-hot ke liye abhi drop kar rahe hain (baad mein target-encoding se add karenge)
# - delay_days: alag regression target hai, classification ke liye use nahi karna
drop_cols = ["project_id", "project_name", "district", "delay_days"]
model_df = df.drop(columns=drop_cols)

print(model_df.shape)

(3000, 18)


In [3]:
# project_type aur state text/category hain -- model ko numbers chahiye
# drop_first=True: agar 5 categories hain, toh sirf 4 columns banenge 
# (5th automatically implied ho jaata hai baaki sab 0 hone se) -- redundancy avoid karta hai
model_df = pd.get_dummies(model_df, columns=["project_type", "state"], drop_first=True)

print(model_df.shape)
model_df.head()

(3000, 27)


,land_area_hectares,affected_families,compensation_disbursed_pct,active_legal_disputes,possession_pct,rnr_progress_pct,stakeholder_responsiveness,dept_historical_delay_rate,days_since_notification,planned_completion_days,...,project_type_Irrigation Canal,project_type_Power Transmission,project_type_Rail Corridor,state_Gujarat,state_Karnataka,state_Madhya Pradesh,state_Maharashtra,state_Odisha,state_Tamil Nadu,state_Uttar Pradesh
0,55.4,295,52.7,1,49.1,31.1,0.51,0.67,338,432,...,True,False,False,False,False,False,False,True,False,False
1,73.5,156,28.3,0,68.2,33.9,0.56,0.67,240,230,...,True,False,False,True,False,False,False,False,False,False
2,109.4,113,24.4,1,43.7,4.9,0.12,0.63,134,285,...,True,False,False,True,False,False,False,False,False,False
3,60.5,113,60.3,1,23.2,3.0,0.41,0.42,141,406,...,True,False,False,False,True,False,False,False,False,False
4,47.3,84,31.3,4,35.5,17.2,0.20,0.68,95,488,...,False,False,True,False,True,False,False,False,False,False


In [4]:
X = model_df.drop(columns=["is_delayed"])
y = model_df["is_delayed"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3000, 26)
y shape: (3000,)


In [5]:
from sklearn.model_selection import train_test_split

# stratify=y: taaki train aur test dono mein delay-rate same proportion mein rahe
# (agar stratify nahi karte, toh kabhi-kabhi train mein zyada delayed aur 
#  test mein kam delayed projects aa sakte hain, jo evaluation ko misleading banata hai)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train delay rate:", y_train.mean(), " Test delay rate:", y_test.mean())

Train shape: (2400, 26)  Test shape: (600, 26)
Train delay rate: 0.37916666666666665  Test delay rate: 0.37833333333333335


In [7]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved all 4 files to data/processed/")

Saved all 4 files to data/processed/
